# Завдання 1:

Для кожної з адміністративних одиниць України завантажити (urllib) тестові структуровані файли, що містять значення VHI-індексу. При зберіганні файлу, до його імені потрібно додати дату та час завантаження. Передбачити повторні запуски скрипту, реалізувати механізм запобігання повторного довантаження та колізії даних

In [17]:
import os
import urllib.request
from datetime import datetime

def download_vhi_data(province_id):
    if not os.path.exists('data'):
        os.makedirs('data')
        
    existing_files = [f for f in os.listdir('data') if f.startswith(f"vhi_id_{province_id}_")]
    if existing_files:
        print(f"Дані для області {province_id} вже існують.")
        return

    url = f"https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?country=UKR&provinceID={province_id}&year1=1981&year2=2024&type=Mean"
    vhi_url = urllib.request.urlopen(url)
    
    date_time = datetime.now().strftime("%Y%m%d%H%M%S")
    filename = f"data/vhi_id_{province_id}_{date_time}.csv"
    
    with open(filename, 'wb') as f:
        f.write(vhi_url.read())
    print(f"VHI для області {province_id} завантажено.")

for i in range(1, 28):
    download_vhi_data(i)

Дані для області 1 вже існують.
Дані для області 2 вже існують.
Дані для області 3 вже існують.
Дані для області 4 вже існують.
Дані для області 5 вже існують.
Дані для області 6 вже існують.
Дані для області 7 вже існують.
Дані для області 8 вже існують.
Дані для області 9 вже існують.
Дані для області 10 вже існують.
Дані для області 11 вже існують.
Дані для області 12 вже існують.
Дані для області 13 вже існують.
Дані для області 14 вже існують.
Дані для області 15 вже існують.
Дані для області 16 вже існують.
Дані для області 17 вже існують.
Дані для області 18 вже існують.
Дані для області 19 вже існують.
Дані для області 20 вже існують.
Дані для області 21 вже існують.
Дані для області 22 вже існують.
Дані для області 23 вже існують.
Дані для області 24 вже існують.
Дані для області 25 вже існують.
Дані для області 26 вже існують.
Дані для області 27 вже існують.


# Завдання 2 & 3:

2. Зчитати завантажені текстові файли у pandas dataframe. Здійснити data cleaning: прибрати зайві стовпці, заповнити пропуски, видалити зайвий текст тощо. Додати стовпчики з назвою та індексом області.
3. Реалізувати процедуру зміни індексів: замініти індекси NOAA на індекси за українською абеткою.

In [18]:
import os
import pandas as pd

def create_and_clean_dataframe(directory_path):
    reindex_map = {
        1: 22, 2: 24, 3: 23, 4: 25, 5: 3, 6: 4, 7: 8, 8: 19, 9: 20, 10: 5,
        11: 9, 12: 26, 13: 10, 14: 11, 15: 12, 16: 13, 17: 14, 18: 15, 19: 16,
        20: 27, 21: 17, 22: 18, 23: 7, 24: 1, 25: 2, 26: 6, 27: 21
    }

    provinces = {
        1: "Вінницька", 2: "Волинська", 3: "Дніпропетровська", 4: "Донецька",
        5: "Житомирська", 6: "Закарпатська", 7: "Запорізька", 8: "Івано-Франківська",
        9: "Київська", 10: "Кіровоградська", 11: "Луганська", 12: "Львівська",
        13: "Миколаївська", 14: "Одеська", 15: "Полтавська", 16: "Рівненська",
        17: "Сумська", 18: "Тернопільська", 19: "Харківська", 20: "Херсонська",
        21: "Хмельницька", 22: "Черкаська", 23: "Чернівецька", 24: "Чернігівська",
        25: "Республіка Крим", 26: "Київ", 27: "Севастополь"
    }

    frames = []

    for filename in os.listdir(directory_path):
        if filename.endswith(".csv"):
            file_path = os.path.join(directory_path, filename)
            
            # ПЕРЕВІРКА: Якщо файл порожній - ігноруємо його
            if os.stat(file_path).st_size == 0:
                print(f"Пропущено порожній файл: {filename}")
                continue

            try:
                old_id = int(filename.split('_')[2])
                new_id = reindex_map.get(old_id)

                # Читаємо файл
                df = pd.read_csv(file_path, index_col=False, header=1)
                
                # Якщо після зчитування виявилося, що в датафреймі немає даних
                if df.empty:
                    continue

                df.columns = df.columns.str.strip().str.lower()
                df = df.rename(columns={col: 'vhi' for col in df.columns if 'vhi' in col})
                
                df['year'] = df['year'].astype(str).str.replace('<tt><pre>', '', regex=False)
                df = df[df['year'] != '']
                df['year'] = pd.to_numeric(df['year'], errors='coerce')
                df['vhi'] = pd.to_numeric(df['vhi'], errors='coerce')

                df['province_id'] = new_id
                df['province_name'] = provinces.get(new_id, "Невідомо")

                frames.append(df.dropna(subset=['year', 'vhi']))
            
            except Exception as e:
                # Якщо файл все одно видає помилку - виводимо її назву і йдемо далі
                print(f"Помилка при читанні файлу {filename}: {e}")
                continue

    return pd.concat(frames, ignore_index=True)

all_data = create_and_clean_dataframe('data')
all_data.sample(12)

,year,week,smn,smt,vci,tci,vhi,province_id,province_name
25021,1990.0,10.0,0.382,289.77,82.83,64.47,73.65,27,Севастополь
41978,2015.0,15.0,0.255,287.68,66.26,18.80,42.53,21,Хмельницька
16414,1996.0,35.0,0.199,296.42,18.91,76.21,47.56,14,Одеська
20540,1990.0,1.0,0.093,269.36,60.60,8.33,34.46,16,Рівненська
41423,2004.0,32.0,0.420,293.85,55.08,61.84,58.46,21,Хмельницька
36454,1995.0,3.0,-1.000,-1.00,-1.00,-1.00,-1.00,2,Волинська
14750,2007.0,35.0,0.222,303.69,41.69,18.50,30.09,13,Миколаївська
23164,1997.0,25.0,0.477,295.03,78.35,69.21,73.78,22,Черкаська
8628,2018.0,49.0,0.050,256.12,28.32,83.12,55.71,10,Кіровоградська
22677,1988.0,6.0,0.041,260.72,30.63,49.93,40.28,22,Черкаська


### **Завдання 4:**

Реалізувати процедуру зміни індексів: в завантажених з NOAA даних області індексуються за англійською абеткою (Province 1 - Cherkasy), потрібно замінити індекси так, щоб області індексувалася за українською абеткою (1 область - Вінницька).

In [22]:
def print_vhi_extremes(df, province_id, start_year, end_year):
    subset = df[(df['province_id'] == province_id) & 
                (df['year'] >= start_year) & 
                (df['year'] <= end_year)]
    
    subset = subset[subset['vhi'] > 0]
    
    print(f"Аналіз VHI для області №{province_id} за період {start_year} - {end_year} рр:")
    
    if not subset.empty:
        max_vhi = subset['vhi'].max()
        min_vhi = subset['vhi'].min()
        print(f"Максимальний VHI: {max_vhi}")
        print(f"Мінімальний VHI: {min_vhi}")
    else:
        print("Помилка: Дані за вказаний період відсутні.")

print_vhi_extremes(all_data, 10, 2000, 2020)

Аналіз VHI для області №10 за період 2000 - 2020 рр:
Максимальний VHI: 84.52
Мінімальний VHI: 16.75


### **Завдання 5: Реалізація допоміжних функцій:**

1. Отримання ряду VHI для вказаної області за конкретний рік.
2. Отримання даних VHI для кількох вибраних областей за вказаний діапазон років.

In [23]:
def get_vhi_for_year(df, province_id, year):
    return df[(df['province_id'] == province_id) & (df['year'] == year)]

def get_vhi_range(df, province_ids, start_year, end_year):
    return df[(df['province_id'].isin(province_ids)) & 
              (df['year'] >= start_year) & 
              (df['year'] <= end_year)]


print("Результат для однієї області за рік (Вінницька - №1):")
display(get_vhi_for_year(all_data, 1, 2010).sample(5))

print("\nРезультат для діапазону областей та років:")
display(get_vhi_range(all_data, [1, 12, 5], 2000, 2015).sample(12))

Результат для однієї області за рік (Вінницька - №1):


,year,week,smn,smt,vci,tci,vhi,province_id,province_name
35028,2010.0,33.0,0.403,298.52,60.06,21.21,40.64,1,Вінницька
35019,2010.0,24.0,0.458,295.22,64.93,60.03,62.48,1,Вінницька
35014,2010.0,19.0,0.339,294.03,54.54,41.73,48.14,1,Вінницька
35010,2010.0,15.0,0.203,289.16,52.41,23.83,38.12,1,Вінницька
35012,2010.0,17.0,0.282,292.34,58.74,30.83,44.79,1,Вінницька



Результат для діапазону областей та років:


,year,week,smn,smt,vci,tci,vhi,province_id,province_name
1323,2007.0,24.0,0.480,298.56,78.15,10.16,44.15,5,Житомирська
12740,2012.0,1.0,0.092,265.21,50.70,35.62,43.16,12,Львівська
12362,2004.0,39.0,0.400,287.58,79.78,36.42,58.10,12,Львівська
12418,2005.0,43.0,0.319,283.57,87.65,15.56,51.61,12,Львівська
34663,2003.0,32.0,0.427,294.98,69.73,68.78,69.25,1,Вінницька
1670,2014.0,7.0,0.084,269.14,54.28,28.15,41.22,5,Житомирська
12901,2015.0,6.0,0.119,270.11,58.97,31.94,45.45,12,Львівська
12804,2013.0,13.0,0.120,268.46,21.31,92.22,56.77,12,Львівська
1410,2009.0,7.0,0.081,267.73,52.97,34.30,43.64,5,Житомирська
12899,2015.0,4.0,0.106,267.79,63.98,29.32,46.65,12,Львівська
